## simple_triton example

This notebook illustrates how to use the simple_triton package to perform inference on tiles from a whole-slide image using a Triton inference server.

Notes for running:
- See https://github.com/PathologyDataScience/simple_triton for details on launching the triton server container and mounting the model repository notebook
- This notebook requires installation of `mil` and `histomics_stream`
- Run this notebook in a container with `--network=host` so that it can reach the Triton container
- Mount your model repository directory to the triton container
- Load the model (below)

In [ ]:
# install large_image with tile sources
!pip install ../../histomics_stream 'large_image[tiff]' \
  scikit_image --find-links https://girder.github.io/large_image_wheels

# install simple_triton
!pip install --upgrade --no-deps --force-reinstall ../../simple_triton

# install mil
!pip install ray
!pip install pyarrow
!pip install ../../mil

## Run client with no GPUs

If running Triton and the client on the same machine, we want to stop the client tensorflow from consuming GPU resources. By default, TensorFlow maps nearly all available GPU memory.

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import tensorflow as tf

assert len(tf.config.list_physical_devices("GPU")) == 0

## Create a histomics stream study

Parameters in this cell are for reading from the whole-slide image (magnification, tile size, tile overlap, mask file).

In [ ]:
from mil.io.utils import study

# slide parameters
batch = 64
magnification = 20
tile = 224
overlap = 0
chunk = 224
mask_threshold = 0.5
wsi_path = (
    "/tf/notebooks/TCGA-AN-A0G0-01Z-00-DX1.BE0BB5DF-DEDA-48D8-B5D8-2735C767F28F.svs"
)
mask_path = "/tf/notebooks/TCGA-AN-A0G0-01Z-00-DX1.BE0BB5DF-DEDA-48D8-B5D8-2735C767F28F.mask.png"

# create a histomics-stream study from a wsi/mask pair
hs_study = study(
    (wsi_path, mask_path), t=(tile, tile), chunk=(tile, tile), target=20, source="exact"
)

## Create and load model

The function `feature_extractor` can be used to create feature extraction models in the model repository. Note - this cell will take time as the model is downloaded, saved, and loaded into triton. 

Here we generate a model, load the model into triton, and verify that the model state is "READY".

Parameters in this stage include the inference server (address), the model (model name, maximum batch size).

In [ ]:
import json
from google.protobuf.json_format import MessageToDict
import numpy as np
from simple_triton.feature_extraction import feature_extractor
from simple_triton.model import load_model
import tritonclient.grpc as grpcclient

# triton parameters
url = "localhost:8001"  # url for grpc access to tirton server
keras_name = "ConvNeXtXLarge"
model_name = f"{keras_name}.tensorflow"  # set model name

# create the model and capture output dimensionality
if not os.path.exists(os.path.join("/tf/notebooks/models", model_name)):
    dimension_output = feature_extractor(
        "/tf/notebooks/models", keras_name, model_name, t=(tile, tile), pool="avg"
    )

# create client
client = grpcclient.InferenceServerClient(url=url, verbose=True)
    
# load tensorflow model
load_model(client, model_name)
    
# display model readiness
client.get_model_repository_index()

# protect subprocesses from inheriting client
del client

## Modify the model configuration

Model resources, optimizations, and batching behavior can be controlled using the model configuration. These can be updated during runtime if the triton container is started with `--model-control-mode=explicit`. The `ConfigBuilder` class can be used to incrementally construct a configuration that optimized model performance.

In [ ]:
from simple_triton.config import ConfigBuilder

# build a basic configuration that increasea maximum batch size
builder = ConfigBuilder(config={"name": model_name, "maxBatchSize": 256})

# increase the number of model instances per GPU to 2
builder.add_instance_group(count=2)

# add automatic mixed precision
builder.add_mixed_precision()

# display configuration
print(builder.config)

# re-load model with new config
client = grpcclient.InferenceServerClient(url=url, verbose=True)
client.load_model(model_name, config=json.dumps(builder.config))
del client

## Run the inference

Parameters here include the number of tiles per batch, the number of workers, and the maximum number of pending inferences per worker.

In [ ]:
from simple_triton.feature_extraction import histomics_stream_inference
from simple_triton.submitter import analyze
import time

# inference parameters
batch = 64
limit = 10  # limit on number of pending requests per worker
workers = 32  # total number of Submitter workers
verbose = True  # set verbose as False

# start timer
start = time.time()

# inference
features, tile_info, times, failures = histomics_stream_inference(
    hs_study,
    model_name,
    url="localhost:8001",
    batch=batch,
    workers=workers,
    limit=limit,
)

# display elapsed time
print(f"Total elapsed time: {time.time()-start}")

# analyze performance
analyze(times)

## Write features to .tfr

Write the extracted features to a TFrecord file and confirm readability.

In [ ]:
from mil.io.reader import read_record, peek
from mil.io.writer import write_record
import tensorflow as tf

# concatenate features
features = np.concatenate(features[0], axis=0)

# create dummy labels
labels = {"labels": np.random.uniform(size=(10))}

# write to tfrecord
write_record(
    "./triton.tfr", features, tile_info, labels, structured=False, precision=tf.float16
)

# get list of .tfr variables for de-serialization
serialized = list(tf.data.TFRecordDataset(["./triton.tfr"]))[0]
variables = peek(serialized)

# verify reading
read_record(serialized, variables, structured=False, precision=tf.float16)